# 📓 Notebook: Extreme Events — Wind and Heavy Precipitation (P95)

### 📌 Objective

This notebook identifies daily extreme weather events using station‑specific
95th percentile (P95) thresholds.

Two extreme event types are generated:
- Extreme Wind
- Heavy Precipitation

The output is a unified, event‑level dataset where each row represents
a single extreme event. Event magnitudes are standardized into a single
`EventValue` column to support consistent downstream analysis.

In [1]:
import pandas as pd
import numpy as np

### 2️⃣ Load daily climate data

We start from the cleaned daily climate dataset, which contains
temperature, precipitation, and wind variables at the station‑day level.
This dataset serves as the raw input for extreme event identification.

In [2]:
climate = pd.read_csv(
    "../Capstone/raw data/fact_Atlantic_Climate.csv",
    parse_dates=["Date/Time"],
    low_memory=False
)

# Normalize column names
climate.columns = climate.columns.str.strip().str.replace(" ", "_")

# Ensure consistent station identifier
climate["ClimateID"] = climate["Climate_ID"].astype(str)

### 3️⃣ EXTREME WIND EVENTS (P95)

Select wind data and calculate P95
Extreme wind events are defined as daily maximum wind gusts that exceed
the station‑specific 95th percentile of historical wind gust values.

Extreme wind events are defined as daily maximum wind gusts that exceed
the station‑specific 95th percentile of historical wind gust values.


In [3]:
wind = climate.dropna(subset=["Spd_of_Max_Gust_(km/h)"]).copy()
wind = wind.rename(columns={"Spd_of_Max_Gust_(km/h)": "MaxGust_kmh"})

p95_wind = (
    wind
    .groupby("ClimateID")["MaxGust_kmh"]
    .quantile(0.95)
    .reset_index()
    .rename(columns={"MaxGust_kmh": "P95_Threshold"})
)

wind_p95 = wind.merge(p95_wind, on="ClimateID", how="left")

In [4]:
extreme_wind = wind_p95[
    wind_p95["MaxGust_kmh"] >= wind_p95["P95_Threshold"]
].copy()

For schema consistency, wind magnitude values are explicitly mapped
to the standardized `EventValue` column.

In [5]:
fact_extreme_wind = extreme_wind[
    ["ClimateID", "Date/Time", "MaxGust_kmh", "P95_Threshold"]
].copy()

fact_extreme_wind["EventType"] = "Extreme Wind"
fact_extreme_wind["ExtremeFlag"] = 1

# ✅ STANDARDIZATION STEP (CRITICAL)
fact_extreme_wind["EventValue"] = fact_extreme_wind["MaxGust_kmh"]

fact_extreme_wind = fact_extreme_wind[
    ["ClimateID", "Date/Time", "EventType", "EventValue", "P95_Threshold", "ExtremeFlag"]
]

### 4️⃣ EXTREME HEAVY PRECIPITATION EVENTS (P95)

 Select precipitation data and calculate P95

 Heavy precipitation extreme events are defined as daily precipitation totals
exceeding the station‑specific 95th percentile.

In [6]:
precip = climate.dropna(subset=["Total_Precip_(mm)"]).copy()
precip = precip.rename(columns={"Total_Precip_(mm)": "DailyPrecip_mm"})

p95_precip = (
    precip
    .groupby("ClimateID")["DailyPrecip_mm"]
    .quantile(0.95)
    .reset_index()
    .rename(columns={"DailyPrecip_mm": "P95_Threshold"})
)

precip_p95 = precip.merge(p95_precip, on="ClimateID", how="left")

In [7]:
extreme_precip = precip_p95[
    precip_p95["DailyPrecip_mm"] >= precip_p95["P95_Threshold"]
].copy()

In [8]:
fact_extreme_precip = extreme_precip[
    ["ClimateID", "Date/Time", "DailyPrecip_mm", "P95_Threshold"]
].copy()

fact_extreme_precip["EventType"] = "Heavy Precipitation"
fact_extreme_precip["ExtremeFlag"] = 1

fact_extreme_precip = fact_extreme_precip.rename(
    columns={"DailyPrecip_mm": "EventValue"}
)

fact_extreme_precip = fact_extreme_precip[
    ["ClimateID", "Date/Time", "EventType", "EventValue", "P95_Threshold", "ExtremeFlag"]
]

## 5️⃣ Combine all extreme events

Extreme wind and heavy precipitation events are combined into a single
event‑level fact table. Each row represents one extreme event.
Multiple events may occur at the same station on the same date.

In [9]:
fact_extreme_all = pd.concat(
    [fact_extreme_wind, fact_extreme_precip],
    ignore_index=True
)

### 6️⃣ Sanity checks

In [10]:
# Ensure schema completeness
fact_extreme_all.isna().sum()

ClimateID        0
Date/Time        0
EventType        0
EventValue       0
P95_Threshold    0
ExtremeFlag      0
dtype: int64

In [11]:
# Check event counts by type
fact_extreme_all.groupby("EventType")["EventValue"].count()

EventType
Extreme Wind           25045
Heavy Precipitation    35333
Name: EventValue, dtype: int64

In [12]:
# Validate threshold logic
assert (
    fact_extreme_all["EventValue"] >= fact_extreme_all["P95_Threshold"]
).all(), "Error: some events fall below P95 threshold"

### 7️⃣ Export final dataset

This file is the single source of truth for extreme event analysis and
is designed to be consumed directly by Power BI and EDA notebooks
without additional corrections.

In [13]:
fact_extreme_all.to_csv(
    "../Capstone/clean data/Fact_Extreme_Events_All.csv",
    index=False
)

### ✅ Final Notes 

- Event magnitudes are standardized using the `EventValue` field:
  - Wind → km/h
  - Precipitation → mm
- Frequency (not intensity aggregation) is the primary analytical focus.
- This dataset is finalized and reproducible.